# Extract transition and reward functions

In [34]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
from value_iteration import value_iteration
from env.user_sim import UserSimEnv
import utils
import mdp_utils
import env.simulate as sim
from sklearn.cluster import KMeans
from morld.mo_pi import MOPolicyIteration
import ipywidgets as widgets
from ipywidgets import interact, fixed
import matplotlib.pyplot as plt

seed = 42
np.random.seed(seed)

In [35]:
MAX_COUNT = 2
NUM_FEATURES = 3
NUM_VALS = 3
NUM_USER_STATES = NUM_VALS**NUM_FEATURES
NUM_CLUSTERS = 5
NUM_COUNT_STATES = (MAX_COUNT+1)**NUM_CLUSTERS
NUM_STATES = NUM_USER_STATES * NUM_COUNT_STATES 
NUM_ACTIONS = 5

In [36]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\'
save_folder = 'functions\\3'
filename = 'simulation_results_2.csv'
df = pd.read_csv(data_folder + filename)

action_file = 'normalized_challenges.csv'
actions = pd.read_csv(data_folder + action_file)


In [37]:
state_features = ['tiredness', 'time_avail', 'pu_state']
reward_signals = ['obs_time', 'obs_liked', 'obs_pu', 'obs_diff']

expert_score_cols = ['score_acceptance', 'score_distraction', 'score_problem_solving', 'score_social_support']
expert_score_matrix = actions[expert_score_cols].values

category_mapping = {"acceptance": 0, "distraction": 1, "problem_solving": 2, "social_support": 3}

actions['category_id'] = actions['category'].map(category_mapping)
action_categories = actions['category_id'].values

df = df.merge(actions[['action_id', 'category_id'] + expert_score_cols], on='action_id', how='left')

df[reward_signals] = df[reward_signals].fillna(0)

NUM_ACTIONS = len(actions)

In [38]:
df['obs_time'] = utils.scale_rewards(df['obs_time'].values, reverse=True) * df['completed']
df['obs_diff'] = utils.scale_rewards(df['obs_diff'].values)
df['obs_liked'] = utils.scale_rewards(df['obs_liked'].values)
df['obs_pu'] = utils.scale_rewards(df['obs_pu'].values)

df.head()

,user_id,timestep,action_id,completed,rw_state,time_avail,motivation,tiredness,obs_diff,pu_state,obs_time,obs_liked,obs_pu,category_id,score_acceptance,score_distraction,score_problem_solving,score_social_support
0,1,1,44,True,1.0,3.0,4.0,3.0,0.428571,4.0,0.285714,0.571429,0.714286,1,0.0,1.000000,0.333333,0.000000
1,1,2,92,False,1.0,3.0,5.0,5.0,0.000000,5.0,0.000000,0.000000,0.000000,3,0.0,0.000000,0.000000,0.666667
2,1,3,101,True,1.0,4.0,5.0,4.0,0.857143,5.0,0.571429,1.000000,1.000000,3,0.0,0.333333,0.000000,0.666667
3,1,4,46,True,2.0,4.0,6.0,4.0,0.571429,6.0,0.428571,0.571429,1.000000,1,0.0,0.666667,0.000000,0.333333
4,1,5,79,False,2.0,5.0,7.0,3.0,0.000000,7.0,0.000000,0.000000,0.000000,3,0.0,0.333333,0.000000,0.666667


### Extract states

In [39]:
def get_feature_representation(feature_vals, num_vals=NUM_VALS):
    # Bin the feature values into discrete categories
    # Maybe use qcute for relative binning?
    binned_vals = pd.cut(feature_vals, bins=num_vals, labels=False)
    return binned_vals

In [40]:
for feature in state_features:
    df['s_' + feature] = get_feature_representation(df[feature])

state_cols = ['s_' + feature for feature in state_features]

df['s_next'] = df.groupby('user_id')[state_cols].shift(-1).values.tolist()
df['s_current'] = df[state_cols].values.tolist()
initial_states = df.groupby('user_id')['s_current'].first().values.tolist()
initial_states_idx = [utils.state_to_idx(state, NUM_FEATURES, 0, num_vals=NUM_VALS) for state in initial_states]
initial_distribution = np.zeros(NUM_USER_STATES)
for idx in initial_states_idx:
    initial_distribution[idx] += 1
initial_distribution /= initial_distribution.sum()
print("Initial state distribution:", initial_distribution)
np.save(os.path.join(data_folder, save_folder, 'initial_distribution.npy'), initial_distribution)

# drop rows where s_next contains NaN values (last row of each user)
df = df[~df['s_next'].apply(lambda x: any(pd.isna(i) for i in x))]
df = df.drop(state_cols + state_features, axis=1)
df = df.drop(['rw_state', 'motivation'], axis=1)



Initial state distribution: [0.04333333 0.04       0.00333333 0.14333333 0.06333333 0.
 0.02       0.00333333 0.         0.11666667 0.04666667 0.00666667
 0.27666667 0.12       0.00666667 0.02       0.00666667 0.00333333
 0.01666667 0.00333333 0.         0.04       0.01666667 0.
 0.00333333 0.         0.        ]


#### Extract counts per category

In [ ]:
cluster_vars = ['obs_diff', 'obs_time', 'obs_liked', 'obs_pu']
cluster_models = {}

# For each action, compute the mean of the cluster variables for completed actions
actions = df[df['completed'] == 1].groupby('action_id')[cluster_vars].mean()
actions['category_id'] = action_categories
actions.head()
actions.to_csv(os.path.join(data_folder, save_folder, 'action_data.csv'), index=True)

actions_clustered, _, cluster_cols = utils.cluster_actions(actions, cluster_vars, num_clusters=NUM_CLUSTERS)

df = df.merge(
    actions_clustered[cluster_cols],
    on='action_id',
    how='left'
)

In [ ]:
cluster_col = 'cluster_all'

for c in range(NUM_CLUSTERS):
    df[f"cat_{c}"] = ((df[cluster_col] == c) & df["completed"]).astype(int)

cat_cols = [f"cat_{c}" for c in range(NUM_CLUSTERS)]

df[cat_cols] = df.groupby("user_id")[cat_cols].cumsum()
df["count"] = df[cat_cols].values.tolist()

df['s_count_next'] = df['count'].apply(lambda x: [min(count - min(x), MAX_COUNT) for count in x])
df['s_count'] = df.groupby('user_id')['s_count_next'].shift(1, fill_value=[0]*NUM_CLUSTERS)

df.drop(cat_cols, axis=1, inplace=True)

#### Extract skill level per category

In [ ]:
# def map_to_tier(normalized_score):
#     """Maps a 0.0-1.0 progress value to discrete skill tiers."""
#     if normalized_score >= 0.75: return 1.0
#     if normalized_score >= 0.50: return 0.67
#     if normalized_score >= 0.25: return 0.33
#     return 0.0


# for col in expert_score_cols:
#     raw_sum_col = f'raw_sum_{col}'
#     df[raw_sum_col] = df.groupby('user_id')[col].transform(lambda x: x.shift(1, fill_value=0).cumsum())
#     df[f'skill_{col}'] = df[raw_sum_col].apply(map_to_tier)

# skill_cols = [f'skill_{col}' for col in expert_score_cols]
# df['skill_current'] = df[skill_cols].values.tolist()
# df['skill_next'] = df.groupby('user_id')[skill_cols].shift(-1).values.tolist()

# df = df.drop(columns=[f'raw_sum_{col}' for col in expert_score_cols])

#### Prepare reward signals

In [ ]:
# reward for skill improvement = reward for completing the action * (sum of skill tiers after - sum of skill tiers before) / max possible increase in skill tiers
# df['r_skill'] = df['completed'] * (df['skill_next'].apply(lambda x: sum(x)) - df['skill_current'].apply(lambda x: sum(x))) / 4
df['r_expert'] = df['completed'] * df[expert_score_cols].sum(axis=1) / 4  # alternative reward based on expert scores for the action
df = df.drop(expert_score_cols, axis=1)
df['r_diversity'] = (1 - (1 / (MAX_COUNT + 1)) * df.apply(lambda row: row['s_count'][row[cluster_col]], axis=1))

df.head(10)

,user_id,timestep,action_id,completed,obs_diff,obs_time,obs_liked,obs_pu,category_id,s_next,...,obs_diff_cluster,obs_time_cluster,obs_liked_cluster,obs_pu_cluster,cluster_all,count,s_count_next,s_count,r_expert,r_diversity
0,1,1,44,True,0.428571,0.285714,0.571429,0.714286,1,"[1.0, 0.0, 1.0]",...,0,3,1,2,4,"[0, 0, 0, 0, 1]","[0, 0, 0, 0, 1]","[0, 0, 0, 0, 0]",0.333333,1.000000
1,1,2,92,False,0.000000,0.000000,0.000000,0.000000,3,"[1.0, 1.0, 1.0]",...,3,3,4,1,2,"[0, 0, 0, 0, 1]","[0, 0, 0, 0, 1]","[0, 0, 0, 0, 1]",0.000000,1.000000
2,1,3,101,True,0.857143,0.571429,1.000000,1.000000,3,"[1.0, 1.0, 2.0]",...,3,0,4,3,3,"[0, 0, 0, 1, 1]","[0, 0, 0, 1, 1]","[0, 0, 0, 0, 1]",0.250000,1.000000
3,1,4,46,True,0.571429,0.428571,0.571429,1.000000,1,"[0.0, 1.0, 2.0]",...,2,4,3,3,0,"[1, 0, 0, 1, 1]","[1, 0, 0, 1, 1]","[0, 0, 0, 1, 1]",0.250000,1.000000
4,1,5,79,False,0.000000,0.000000,0.000000,0.000000,3,"[1.0, 1.0, 2.0]",...,3,3,3,2,2,"[1, 0, 0, 1, 1]","[1, 0, 0, 1, 1]","[1, 0, 0, 1, 1]",0.000000,1.000000
5,1,6,15,True,0.142857,0.857143,0.428571,1.000000,0,"[1.0, 2.0, 2.0]",...,0,0,1,1,0,"[2, 0, 0, 1, 1]","[2, 0, 0, 1, 1]","[1, 0, 0, 1, 1]",0.166667,0.666667
6,1,7,77,True,0.714286,0.714286,0.428571,1.000000,2,"[1.0, 1.0, 2.0]",...,1,2,3,3,1,"[2, 1, 0, 1, 1]","[2, 1, 0, 1, 1]","[2, 0, 0, 1, 1]",0.166667,1.000000
7,1,8,51,True,0.285714,0.428571,0.714286,1.000000,1,"[0.0, 1.0, 2.0]",...,2,1,2,2,1,"[2, 2, 0, 1, 1]","[2, 2, 0, 1, 1]","[2, 1, 0, 1, 1]",0.333333,0.666667
8,1,9,62,True,0.571429,0.000000,0.285714,1.000000,2,"[0.0, 1.0, 2.0]",...,0,3,3,4,4,"[2, 2, 0, 1, 2]","[2, 2, 0, 1, 2]","[2, 2, 0, 1, 1]",0.166667,0.666667
9,1,10,30,True,0.857143,0.142857,1.000000,1.000000,1,"[1.0, 0.0, 2.0]",...,3,3,2,2,1,"[2, 3, 0, 1, 2]","[2, 2, 0, 1, 2]","[2, 2, 0, 1, 2]",0.333333,0.333333


## Learn transition and reward functions

### Estimate completion probability
For each $(u,a)$ pair we want to estimate the completion probability. To do this we cluster the actions based on perceived usefulness.

In [ ]:
df['s_idx'] = df['s_current'].apply(lambda s: utils.state_to_idx(tuple(s), num_feats=NUM_FEATURES, num_counts=0, num_vals=NUM_VALS, max_count=0))
df['c_idx'] = df['s_count'].apply(lambda c: utils.state_to_idx(tuple(c), num_feats=NUM_CLUSTERS, num_counts=0, num_vals=MAX_COUNT+1, max_count=0))
df['sp_idx'] = df['s_next'].apply(lambda sp: utils.state_to_idx(tuple(sp), num_feats=NUM_FEATURES, num_counts=0, num_vals=NUM_VALS, max_count=0)).astype(int)
df['s_full_idx'] = df.apply(lambda row: utils.state_to_idx(tuple(row['s_current']) + tuple(row['s_count']), num_feats=NUM_FEATURES, num_counts=NUM_CLUSTERS, num_vals=NUM_VALS, max_count=MAX_COUNT), axis=1)

In [ ]:
samples = df[['user_id', 'timestep', 'action_id', 'completed', 'obs_diff', 'obs_time',
       'obs_liked', 'obs_pu', 'category_id',
       'obs_diff_cluster', 'obs_time_cluster', 'obs_liked_cluster',
       'obs_pu_cluster', 'cluster_all',
       'r_expert', 'r_diversity', 's_idx', 'c_idx', 'sp_idx', 's_full_idx']]
samples.to_csv(os.path.join(data_folder, save_folder, 'samples.csv'), index=False)

In [ ]:
obj_to_cluster = {
    "obs_time": "obs_time_cluster",
    "obs_liked": "obs_liked_cluster",
    "obs_pu": "obs_pu_cluster",
    "r_diversity": "cluster_all",
    "r_expert": "action_id",
}
nO = len(obj_to_cluster)
P_c_clustered_all_actions = mdp_utils.compute_completion_probabilities_clustered(df, 'cluster_all', NUM_USER_STATES, NUM_ACTIONS, num_clusters=NUM_CLUSTERS, use_clusters=False)
R_clustered_all_actions = mdp_utils.compute_rewards_clustered(df, 
                                          P_c_clustered_all_actions, 
                                          obj_to_cluster, 
                                          NUM_USER_STATES, 
                                          NUM_COUNT_STATES, 
                                          NUM_ACTIONS, 
                                          nO, 
                                          action_categories,
                                          num_clusters=NUM_CLUSTERS,
                                          count_cluster=cluster_col,
                                          use_clusters=False)
P_clustered_all_actions = mdp_utils.compute_transition_probabilities_clustered(df, 
                                                                    NUM_USER_STATES, 
                                                                    NUM_ACTIONS, 
                                                                    'cluster_all', 
                                                                    num_clusters=NUM_CLUSTERS, 
                                                                    use_clusters=False)

print("P shape:", P_clustered_all_actions.shape)  # Should be (nU, nA, nU)
print("R shape:", R_clustered_all_actions.shape)  # Should be (nU, nC, nA, nO)

env_cl_all = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = 5, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=P_clustered_all_actions, 
                 completion_probs=P_c_clustered_all_actions,
                 reward_matrix=R_clustered_all_actions, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

P shape: (27, 104, 27)
R shape: (27, 243, 104, 5)


In [ ]:
same_cluster = {
    "obs_time": "category_id",
    "obs_liked": "category_id",
    "obs_pu": "category_id",
    "r_diversity": "category_id",
    "r_expert": "category_id",
}
action_categories = actions[cluster_col].values
nO = len(obj_to_cluster)
n_clusters = 4
P_c_clustered = mdp_utils.compute_completion_probabilities_clustered(df, 'category_id', NUM_USER_STATES, NUM_ACTIONS, num_clusters=n_clusters, use_clusters=True)
R_clustered = mdp_utils.compute_rewards_clustered(df, 
                                          P_c_clustered, 
                                          same_cluster, 
                                          NUM_USER_STATES, 
                                          NUM_COUNT_STATES, 
                                          NUM_ACTIONS, 
                                          nO, 
                                          action_categories,
                                          num_clusters=n_clusters,
                                          count_cluster='category_id',
                                          use_clusters=True)
P_clustered = mdp_utils.compute_transition_probabilities_clustered(df, 
                                                                    NUM_USER_STATES, 
                                                                    NUM_ACTIONS, 
                                                                    'category_id', 
                                                                    num_clusters=n_clusters, 
                                                                    use_clusters=True)

print("P shape:", P_clustered.shape)  # Should be (nU, nA, nU)
print("R shape:", R_clustered.shape)  # Should be (nU, nC, nA, nO)

env_cl_cat = mo_gym.make('user_env', num_actions=n_clusters, 
                 num_objectives = 5, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=P_clustered, 
                 completion_probs=P_c_clustered,
                 reward_matrix=R_clustered, 
                 action_categories=np.array([0,1,2,3]),
                 MAX_COUNT=MAX_COUNT)

P shape: (27, 4, 27)
R shape: (27, 243, 4, 5)


In [ ]:
same_cluster = {
    "obs_time": "cluster_all",
    "obs_liked": "cluster_all",
    "obs_pu": "cluster_all",
    "r_expert": "cluster_all",
    "r_diversity": "cluster_all",
}
nO = len(same_cluster)
n_clusters = 5
P_c_clustered = mdp_utils.compute_completion_probabilities_clustered(df, 'cluster_all', NUM_USER_STATES, NUM_ACTIONS, num_clusters=n_clusters, use_clusters=True)
R_clustered = mdp_utils.compute_rewards_clustered(df, 
                                          P_c_clustered, 
                                          same_cluster, 
                                          NUM_USER_STATES, 
                                          NUM_COUNT_STATES, 
                                          NUM_ACTIONS, 
                                          nO, 
                                          action_categories,
                                          num_clusters=n_clusters,
                                          count_cluster=cluster_col,
                                          use_clusters=True)
P_clustered = mdp_utils.compute_transition_probabilities_clustered(df, 
                                                                    NUM_USER_STATES, 
                                                                    NUM_ACTIONS, 
                                                                    'cluster_all', 
                                                                    num_clusters=n_clusters, 
                                                                    use_clusters=True)

print("P shape:", P_clustered.shape)  # Should be (nU, nA, nU)
print("R shape:", R_clustered.shape)  # Should be (nU, nC, nA, nO)

env_cl = mo_gym.make('user_env', num_actions=n_clusters, 
                 num_objectives = nO, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=P_clustered, 
                 completion_probs=P_c_clustered,
                 reward_matrix=R_clustered, 
                 action_categories=np.array([i for i in range(n_clusters)]),
                 MAX_COUNT=MAX_COUNT)

P shape: (27, 5, 27)
R shape: (27, 243, 5, 5)


In [ ]:
P_c = mdp_utils.compute_completion_probabilities(df, NUM_USER_STATES, NUM_ACTIONS)
R = mdp_utils.compute_rewards(df, P_c, NUM_USER_STATES, NUM_COUNT_STATES, NUM_ACTIONS, nO, obj_to_cluster.keys())
P = mdp_utils.compute_transition_probabilities(df, NUM_USER_STATES, NUM_ACTIONS)
print("P shape:", P.shape)  # Should be (nU, nA, nU)
print("R shape:", R.shape)  # Should be (nU, nC, nA, nO)

env_all = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = 5, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=P, 
                 completion_probs=P_c,
                 reward_matrix=R, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

P shape: (27, 104, 27)
R shape: (27, 243, 104, 5)


### Learn transition function
Our state consists of a stochastic part: tiredness, time available, and motivation. We thus need to learn the transition probabilities for these states and the actions.

$T(s,a,s') = \frac{N(s,a,s')}{N(s,a)}$ 

In [ ]:
# save_folder = data_folder + save_folder
# if not os.path.exists(save_folder):
#     os.makedirs(save_folder)

# filename = os.path.join(save_folder, 'reward_matrix.npy')
# np.save(filename, reward_matrix)
# filename = os.path.join(save_folder, 'transition_probs.npy')
# np.save(filename, transition_probs_array)
# filename = os.path.join(save_folder, 'completion_probs.npy')
# np.save(filename, completion_probs_array)

In [ ]:
# random simulation for each environment
NUM_USERS = 100
NUM_TIMESTEPS = 28
NUM_OBJECTIVES = 5

simulation_results = []
policy_names = []

envs = [env_all, env_cl_all, env_cl_cat, env_cl]
env_names = ['All Actions', 'Clustered All Actions', 'Clustered by Category', 'Clustered by All Variables']
for env, name in zip(envs, env_names):
    print(f"Simulating environment: {name}")
    weights = 1/env.unwrapped.nO * np.ones(env.unwrapped.nO)  # equal weights for all objectives
    vi_agent = MOPolicyIteration(id=0, env=env.unwrapped, weights=weights, gamma=0.7, scalarization='linear')
    vi_agent.train(max_eval_iters=5)
    vi_agent.evaluate()
    policy = vi_agent.policy_table
    print(f"Policy for {name}:", policy)
    print(f"Expected return for {name}:", vi_agent.expected_return)
    sim_res = sim.simulate(env, NUM_USERS, policy, verbose=False, T=NUM_TIMESTEPS, num_vals=NUM_VALS, max_count=MAX_COUNT)
    simulation_results.append(sim_res)
    policy_names.append(name)
objectives = ['obs_time', 'obs_liked', 'obs_pu', 'r_diversity', 'r_expert']
sim.interactive_plot_objective(simulation_results, policy_names, objectives, NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)

Simulating environment: All Actions


Running policy iteration with weights: [0.2 0.2 0.2 0.2 0.2]
Policy converged after 6 iterations.
Policy for All Actions: [10 80 38 ... 61 61 61]
Expected return for All Actions: [2.04031435 2.77502829 2.78812034 3.01949897 0.96913701]
Simulating environment: Clustered All Actions
Running policy iteration with weights: [0.2 0.2 0.2 0.2 0.2]
Policy converged after 5 iterations.
Policy for Clustered All Actions: [ 46  46  46 ... 101 101 101]
Expected return for Clustered All Actions: [1.37292281 1.26346209 1.59351924 1.71305343 0.65037499]
Simulating environment: Clustered by Category
Running policy iteration with weights: [0.2 0.2 0.2 0.2 0.2]
Policy converged after 7 iterations.
Policy for Clustered by Category: [1 1 1 ... 2 2 2]
Expected return for Clustered by Category: [1.03635397 1.37230772 1.73759569 2.10183126 0.57680033]
Simulating environment: Clustered by All Variables
Running policy iteration with weights: [0.2 0.2 0.2 0.2 0.2]
Policy converged after 5 iterations.
Policy for 

interactive(children=(Dropdown(description='Objective:', options=(('obs_time', 0), ('obs_liked', 1), ('obs_pu'…